# Harris Corner Detection

### Objective
In this assignment, you will implement the Harris Corner Detection algorithm from scratch. This algorithm is fundamental in computer vision for detecting corners in images, which are useful features for image matching, object recognition, and tracking.
### Background
The Harris Corner Detection algorithm works by analyzing the local changes in image intensity to identify points that have large intensity changes in multiple directions. These points are likely to be corners. The algorithm computes a response function based on the eigenvalues of the second-moment matrix (also called the Harris matrix) for each pixel.

You will be given a skeleton code with missing parts that you need to implement. The completed code should detect corners in an input image and mark them with red dots.
You need to implement key functions:

Gaussian kernel generation

Image gradient computation

Harris matrix construction

Corner response calculation

Non-maximum suppression and thresholding

In [5]:
from __future__ import division
import numpy as np
import cv2
import matplotlib.pyplot as plt
from numpy import linalg as LA
from tqdm import tqdm

# Helper function for generating Gaussian kernel
def MyGaussianKernel(height, width, sigma):
    """
    Generate a Gaussian kernel of the specified dimensions and standard deviation.
    """
    # TODO: Implement this function
    # HINT: Use the Gaussian function formula to generate the kernel
    # The kernel should be normalized so that its sum equals 1
    kernel = np.zeros((height, width), dtype=np.float64)
    center_x = height // 2
    center_y = width // 2

    for i in range(height):
        for j in range(width):
            x = i - center_x
            y = j - center_y
            kernel[i, j] = np.exp(-(x**2 + y**2) / (2 * sigma**2))

    # Normalize
    kernel = kernel / np.sum(kernel)
    return kernel

# Helper function for downsampling an image
def DownsampleMyImage(image, factor=2):
    """
    Downsample an image by the given factor.
    """
    # TODO: Implement this function
    # HINT: You can use simple indexing or OpenCV's resize function
    return image[::factor, ::factor]

def getGradients(image):
    """
    Compute the image gradients in x and y directions.
    Returns dx and dy - the gradients in x and y directions
    """
    # TODO: Implement this function
    # HINT: Compute the gradients by taking the difference between adjacent pixels
    # For dx, compute I(x+1,y) - I(x,y)
    # For dy, compute I(x,y+1) - I(x,y)
    rows, cols = image.shape
    dx = np.zeros_like(image, dtype=np.float64)
    dy = np.zeros_like(image, dtype=np.float64)

    for i in range(rows - 1):
        for j in range(cols - 1):
            dx[i, j] = image[i, j+1] - image[i, j]
            dy[i, j] = image[i+1, j] - image[i, j]

    return dx, dy

def HarrisCornerDetection(image, kSize, sigma):
    """
    Implement the Harris Corner Detection algorithm.

    Args:
        image: Input image (grayscale)
        kSize: Size of the window for gradient computation
        sigma: Standard deviation for Gaussian kernel

    Returns:
        RMatrix: Response matrix with corner scores
    """
    # Convert to grayscale if the image is colored
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    image = np.float64(image)

    # Generate Gaussian kernel
    kernel = MyGaussianKernel(kSize, kSize, sigma)

    # Initialize response matrix
    RMatrix = np.zeros(image.shape, dtype=np.float64)

    kCenter = kSize // 2
    k = 0.04  # Harris constant


    # Iterate over the image
    for i in tqdm(range(kCenter, image.shape[0] - kCenter)):
        for j in range(kCenter, image.shape[1] - kCenter):

          # Extract local window
            SubImage = image[i-kCenter:i+kCenter+1, j-kCenter:j+kCenter+1]

            # Gradients
            Ix, Iy = getGradients(SubImage)

            # TODO: Compute gradients for the subregion
            Ix, Iy = getGradients(SubImage)

            # TODO: Compute products of gradients
            # HINT: Ix2 = Ix^2, Iy2 = Iy^2, Ixy = Ix*Iy
            Ix2 = Ix * Ix
            Iy2 = Iy * Iy
            Ixy = Ix * Iy

            # TODO: Apply Gaussian weights to the products
            # HINT: Multiply each product by the corresponding weight from the Gaussian kernel
            Sx2 = np.sum(kernel * Ix2)
            Sy2 = np.sum(kernel * Iy2)
            Sxy = np.sum(kernel * Ixy)

            # TODO: Construct the Harris matrix M
            # M = [sum(Ix2)  sum(Ixy)]
            #     [sum(Ixy)  sum(Iy2)]
            M = np.array([[Sx2, Sxy],
                          [Sxy, Sy2]])

            # TODO: Compute the Harris response R
            # R = det(M) - k * (trace(M))^2
            # where k is typically 0.04-0.06
            # HINT: You can use eigenvalues to compute this
            detM = np.linalg.det(M)
            traceM = np.trace(M)

            # Store the response value
            R = detM - k * (traceM ** 2)
            RMatrix[i, j] = R

    return RMatrix

def NMS_and_Threshold(image, RMatrix, thresholds, window_size=3):
    """
    Perform Non-Maximum Suppression and thresholding to find corner points.

    Args:
        image: Original colored image
        RMatrix: Harris response matrix
        thresholds: List of threshold values for corner detection
        window_size: Size of the window for NMS

    Returns:
        images: List of images with marked corners for different thresholds
    """
    # TODO: Implement this function
    # HINT:
    # 1. Create copies of the original image for each threshold
    # 2. For each pixel, check if it's a local maximum in its neighborhood
    # 3. If it is and its R value is above the threshold, mark it as a corner
    # 4. Return the list of images with marked corners
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    images = []
    offset = window_size // 2

    for T in thresholds:
        output = image.copy()

        for i in range(offset, RMatrix.shape[0] - offset):
            for j in range(offset, RMatrix.shape[1] - offset):

                window = RMatrix[i-offset:i+offset+1, j-offset:j+offset+1]

                # Check local maximum
                if RMatrix[i, j] == np.max(window) and RMatrix[i, j] > T:
                    # Mark corner (red dot)
                    cv2.circle(output, (j, i), 2, (0, 0, 255), -1)

        images.append(output)

    return images

def main():
    # Load the image
    image_path = "cameraman.tif"
    image = cv2.imread(image_path, 1)

    # Parameters
    kSize = 7  # Window size
    sigma = 1.5  # Standard deviation for Gaussian kernel
    thresholds = [100, 1000, 10000]  # Different thresholds for corner detection

    # Perform Harris Corner Detection
    RMatrix = HarrisCornerDetection(image, kSize, sigma)

    # Save the response matrix
    np.save("RMatrix.npy", RMatrix)

    # Perform NMS and thresholding
    result_images = NMS_and_Threshold(image, RMatrix, thresholds)

    # Save the results
    for i, img in enumerate(result_images):
        cv2.imwrite(f"corners_T{i+1}.png", img)

    print("Harris Corner Detection completed successfully!")

if __name__ == "__main__":
    main()

100%|██████████| 250/250 [00:09<00:00, 26.93it/s]


Harris Corner Detection completed successfully!
